In [ ]:
!pip install -q transformers accelerate sentencepiece pandas scikit-learn

import json
import torch
import pandas as pd

from pathlib import Path
from google.colab import drive, files
from transformers import AutoTokenizer, AutoModelForCausalLM

drive.mount("/content/drive")

# Load the original development dataset
if not Path("/content/xed_dev.tsv").exists():
    files.upload()

df = pd.read_csv("/content/xed_dev.tsv", sep="\t")
df = df.reset_index(drop=True)

assert len(df) == 954

print("Development examples:", len(df))

# GPU setup
if not torch.cuda.is_available():
    raise RuntimeError("Enable T4 GPU in Colab first.")

print("GPU:", torch.cuda.get_device_name(0))

# Load Granite
MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
).to("cuda")

model.eval()

print("Granite loaded:", MODEL_NAME)
print("Device:", model.device)

Mounted at /content/drive


Saving xed_dev.tsv to xed_dev.tsv
Development examples: 954
GPU: Tesla T4


config.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Secondary model loaded: bigscience/mt0-base


In [ ]:
EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

examples_path = Path(
    "/content/drive/MyDrive/qwen_five_shot_examples.tsv"
)

assert examples_path.exists(), (
    "Cannot find the five fixed Qwen training examples."
)

five_examples = pd.read_csv(examples_path, sep="\t")

assert len(five_examples) == 5

print("Loaded the same five examples used by Qwen.")


def create_zero_shot_prompt(sentence):
    return f"""Identify the emotions in this sentence.

Possible emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Return only a JSON array of the emotions expressed in the sentence.
Do not explain your answer.

Sentence: {sentence}
Answer:"""


def create_five_shot_prompt(sentence, language):

    column = (
        "english_text"
        if language == "english"
        else "es_text"
    )

    prompt = """Identify the emotions in this sentence.

Possible emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Return only a JSON array of the emotions expressed in the sentence.
Do not explain your answer.

"""

    for _, example in five_examples.iterrows():

        labels = json.loads(example["emotion_labels"])

        prompt += (
            f"Sentence: {example[column]}\n"
            f"Answer: {json.dumps(labels)}\n\n"
        )

    prompt += f"Sentence: {sentence}\nAnswer:"

    return prompt


def parse_prediction(output):

    try:
        labels = json.loads(output.strip())

        if not isinstance(labels, list):
            return [], True

        if not all(isinstance(x, str) for x in labels):
            return [], True

        labels = [x.strip().lower() for x in labels]

        if not all(x in EMOTIONS for x in labels):
            return [], True

        return list(dict.fromkeys(labels)), False

    except (json.JSONDecodeError, TypeError, AttributeError):
        return [], True

Loaded the same five examples used by Qwen.


In [ ]:
SYSTEM_INSTRUCTION = """You are a multi-label emotion classifier.

Allowed emotion labels:
anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Identify the emotions expressed in the user's sentence.

Return exactly one valid JSON array of allowed emotion labels.
For example: ["joy", "trust"] or [].

Do not invent other emotion labels.
Do not include explanations.
Do not generate additional examples, sentences, or answers.
"""


def create_granite_messages(sentence, language, shot_type):

    column = "english_text" if language == "english" else "es_text"

    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION}
    ]

    if shot_type == "five":

        for _, example in five_examples.iterrows():

            labels = json.loads(example["emotion_labels"])

            messages.append({
                "role": "user",
                "content": example[column]
            })

            messages.append({
                "role": "assistant",
                "content": json.dumps(labels)
            })

    messages.append({
        "role": "user",
        "content": sentence
    })

    return messages

In [ ]:
from pathlib import Path
import pandas as pd
import json
import torch

def run_granite_full(language, shot_type, batch_size=4):

    assert language in ["english", "spanish"]
    assert shot_type in ["zero", "five"]

    column = "english_text" if language == "english" else "es_text"

    save_path = Path(
        f"/content/drive/MyDrive/"
        f"granite_v2_{shot_type}_shot_{language}_full_dev.csv"
    )

    if save_path.exists():
        previous = pd.read_csv(save_path)
        results = previous.to_dict("records")
        completed = set(previous["row_index"].astype(int))
    else:
        results = []
        completed = set()

    remaining = [i for i in df.index if i not in completed]

    print(f"\nGRANITE V2 {language.upper()} {shot_type.upper()}-SHOT")
    print("Already completed:", len(completed))
    print("Remaining:", len(remaining))

    def save_checkpoint():
        if results:
            pd.DataFrame(results).to_csv(save_path, index=False)

    try:
        for start in range(0, len(remaining), batch_size):

            indices = remaining[start:start + batch_size]

            prompts = []

            for i in indices:
                sentence = df.loc[i, column]

                messages = create_granite_messages(
                    sentence,
                    language,
                    shot_type
                )

                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    thinking=False
                )

                prompts.append(prompt)

            inputs = tokenizer(
                prompts,
                padding=True,
                return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=35,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )

            input_length = inputs["input_ids"].shape[1]

            raw_outputs = tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )

            for i, raw in zip(indices, raw_outputs):

                raw = raw.strip()
                predicted, malformed = parse_prediction(raw)

                results.append({
                    "row_index": int(i),
                    "sentence": df.loc[i, column],
                    "actual": df.loc[i, "emotion_labels"],
                    "predicted": json.dumps(predicted),
                    "raw_output": raw,
                    "malformed": malformed
                })

            # Save every five batches
            if (start // batch_size + 1) % 5 == 0:
                save_checkpoint()
                print(f"Saved {len(results)}/{len(df)}")

    finally:
        save_checkpoint()

    print("\nCompleted:", len(results))
    print("Saved to:", save_path)

    return results

In [ ]:
granite_zero_en_full = run_granite_full(
    "english", "zero", batch_size=4
)


GRANITE V2 ENGLISH ZERO-SHOT
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954

Completed: 954
Saved to: /content/drive/MyDrive/granite_v2_zero_shot_english_full_dev.csv


In [ ]:
granite_zero_es_full = run_granite_full(
    "spanish", "zero", batch_size=4
)


GRANITE V2 SPANISH ZERO-SHOT
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954

Completed: 954
Saved to: /content/drive/MyDrive/granite_v2_zero_shot_spanish_full_dev.csv


In [ ]:
granite_five_en_full = run_granite_full(
    "english", "five", batch_size=4
)


GRANITE V2 ENGLISH FIVE-SHOT
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954

Completed: 954
Saved to: /content/drive/MyDrive/granite_v2_five_shot_english_full_dev.csv


In [ ]:
granite_five_es_full = run_granite_full(
    "spanish", "five", batch_size=4
)


GRANITE V2 SPANISH FIVE-SHOT
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954

Completed: 954
Saved to: /content/drive/MyDrive/granite_v2_five_shot_spanish_full_dev.csv


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

def evaluate_granite(language, shot_type):

    path = (
        f"/content/drive/MyDrive/"
        f"granite_v2_{shot_type}_shot_{language}_full_dev.csv"
    )

    data = pd.read_csv(path)

    assert len(data) == 954
    assert data["row_index"].nunique() == 954

    actual = data["actual"].apply(json.loads).tolist()
    predicted = data["predicted"].apply(json.loads).tolist()

    mlb = MultiLabelBinarizer(classes=EMOTIONS)

    y_true = mlb.fit_transform(actual)
    y_pred = mlb.transform(predicted)

    malformed_rate = (
        data["malformed"]
        .astype(str)
        .str.lower()
        .eq("true")
        .mean()
    )

    metrics = {
        "model": "Granite 3.3 2B Instruct",
        "language": language,
        "method": shot_type + "-shot",
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "jaccard": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match": accuracy_score(y_true, y_pred),
        "malformed_rate": malformed_rate
    }

    print(f"\nGRANITE {language.upper()} {shot_type.upper()}-SHOT")

    for key, value in metrics.items():
        print(f"{key}: {value}")

    print("\nPer-emotion classification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=EMOTIONS,
        zero_division=0
    ))

    return metrics


all_granite_metrics = [
    evaluate_granite("english", "zero"),
    evaluate_granite("spanish", "zero"),
    evaluate_granite("english", "five"),
    evaluate_granite("spanish", "five")
]

granite_summary = pd.DataFrame(all_granite_metrics)

granite_summary.to_csv(
    "/content/drive/MyDrive/granite_v2_full_dev_metrics.csv",
    index=False
)

print("\nAll Granite metrics saved!")


GRANITE ENGLISH ZERO-SHOT
model: Granite 3.3 2B Instruct
language: english
method: zero-shot
micro_f1: 0.22853957636566333
macro_f1: 0.20530775266947215
jaccard: 0.11862334032145352
precision: 0.45964125560538116
recall: 0.15207715133531158
hamming_loss: 0.18134171907756813
exact_match: 0.05870020964360587
malformed_rate: 0.05555555555555555

Per-emotion classification report:
              precision    recall  f1-score   support

       anger       0.53      0.36      0.43       248
anticipation       0.27      0.03      0.06       216
     disgust       0.63      0.10      0.17       119
        fear       0.54      0.22      0.32       143
         joy       0.48      0.09      0.15       162
     sadness       0.64      0.18      0.28       138
    surprise       0.30      0.07      0.11       156
       trust       0.21      0.08      0.12       166

   micro avg       0.46      0.15      0.23      1348
   macro avg       0.45      0.14      0.21      1348
weighted avg       0.44

In [ ]:
from google.colab import files, drive
from pathlib import Path
import pandas as pd
import json
import torch

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

if not Path("/content/xed_test.tsv").exists():
    files.upload()  # Upload xed_test.tsv

test_df = pd.read_csv("/content/xed_test.tsv", sep="\t")
test_df = test_df.reset_index(drop=True)

assert len(test_df) == 483

required = [
    "model",
    "tokenizer",
    "five_examples",
    "create_granite_messages",
    "parse_prediction"
]

for name in required:
    print(name, "OK" if name in globals() else "MISSING")

print("Test examples:", len(test_df))
print("Model device:", model.device)
print("Model:", model.config._name_or_path)

model OK
tokenizer OK
five_examples OK
create_granite_messages OK
parse_prediction OK
Test examples: 483
Model device: cuda:0
Model: ibm-granite/granite-3.3-2b-instruct


In [ ]:
def run_granite_test(language, shot_type, batch_size=4):

    assert language in ["english", "spanish"]
    assert shot_type in ["zero", "five"]

    column = "english_text" if language == "english" else "es_text"

    save_path = Path(
        f"/content/drive/MyDrive/"
        f"granite_v2_{shot_type}_shot_{language}_TEST.csv"
    )

    if save_path.exists():
        previous = pd.read_csv(save_path)
        assert previous["row_index"].is_unique
        results = previous.to_dict("records")
        completed = set(previous["row_index"].astype(int))
    else:
        results = []
        completed = set()

    remaining = [
        i for i in test_df.index
        if i not in completed
    ]

    print(f"\nGRANITE {language.upper()} {shot_type.upper()}-SHOT TEST")
    print("Already completed:", len(completed))
    print("Remaining:", len(remaining))

    def save_checkpoint():
        if results:
            pd.DataFrame(results).to_csv(save_path, index=False)

    try:
        for start in range(0, len(remaining), batch_size):

            indices = remaining[start:start + batch_size]
            prompts = []

            for i in indices:
                sentence = test_df.loc[i, column]

                messages = create_granite_messages(
                    sentence,
                    language,
                    shot_type
                )

                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    thinking=False
                )

                prompts.append(prompt)

            inputs = tokenizer(
                prompts,
                padding=True,
                return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=35,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )

            input_length = inputs["input_ids"].shape[1]

            raw_outputs = tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )

            for i, raw in zip(indices, raw_outputs):

                raw = raw.strip()
                predicted, malformed = parse_prediction(raw)

                actual = json.loads(
                    test_df.loc[i, "emotion_labels"]
                )

                results.append({
                    "row_index": int(i),
                    "sentence": test_df.loc[i, column],
                    "actual": json.dumps(actual),
                    "predicted": json.dumps(predicted),
                    "raw_output": raw,
                    "malformed": malformed
                })

            if (start // batch_size + 1) % 5 == 0:
                save_checkpoint()
                print(f"Saved {len(results)}/{len(test_df)}")

    finally:
        save_checkpoint()

    print("Completed:", len(results))
    print("Saved to:", save_path)

    return results

In [ ]:
granite_test_zero_en = run_granite_test(
    "english", "zero", batch_size=4
)


GRANITE ENGLISH ZERO-SHOT TEST
Already completed: 0
Remaining: 483


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/granite_v2_zero_shot_english_TEST.csv


In [ ]:
granite_test_zero_es = run_granite_test(
    "spanish", "zero", batch_size=4
)


GRANITE SPANISH ZERO-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/granite_v2_zero_shot_spanish_TEST.csv


In [ ]:
granite_test_five_en = run_granite_test(
    "english", "five", batch_size=4
)


GRANITE ENGLISH FIVE-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/granite_v2_five_shot_english_TEST.csv


In [ ]:
granite_test_five_es = run_granite_test(
    "spanish", "five", batch_size=4
)


GRANITE SPANISH FIVE-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/granite_v2_five_shot_spanish_TEST.csv


In [ ]:
import pandas as pd
import json
from pathlib import Path

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

base = Path("/content/drive/MyDrive")

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

granite_test_files = {
    "English zero-shot": "granite_v2_zero_shot_english_TEST.csv",
    "Spanish zero-shot": "granite_v2_zero_shot_spanish_TEST.csv",
    "English five-shot": "granite_v2_five_shot_english_TEST.csv",
    "Spanish five-shot": "granite_v2_five_shot_spanish_TEST.csv"
}

results = []

for experiment, filename in granite_test_files.items():

    data = pd.read_csv(base / filename)

    assert len(data) == 483
    assert data["row_index"].nunique() == 483

    actual = data["actual"].apply(json.loads).tolist()
    predicted = data["predicted"].apply(json.loads).tolist()

    mlb = MultiLabelBinarizer(classes=EMOTIONS)

    y_true = mlb.fit_transform(actual)
    y_pred = mlb.transform(predicted)

    metrics = {
        "Experiment": experiment,
        "Micro-F1": f1_score(
            y_true, y_pred,
            average="micro", zero_division=0
        ),
        "Macro-F1": f1_score(
            y_true, y_pred,
            average="macro", zero_division=0
        ),
        "Jaccard": jaccard_score(
            y_true, y_pred,
            average="samples", zero_division=0
        ),
        "Precision": precision_score(
            y_true, y_pred,
            average="micro", zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred,
            average="micro", zero_division=0
        ),
        "Hamming Loss": hamming_loss(y_true, y_pred),
        "Exact Match": accuracy_score(y_true, y_pred),
        "Malformed Rate": (
            data["malformed"]
            .astype(str)
            .str.lower()
            .eq("true")
            .mean()
        )
    }

    results.append(metrics)

    print("\n" + "=" * 55)
    print("GRANITE", experiment.upper())
    print("=" * 55)

    print(classification_report(
        y_true,
        y_pred,
        target_names=EMOTIONS,
        zero_division=0
    ))

granite_final_metrics = pd.DataFrame(results)

print("\nFINAL GRANITE TEST RESULTS")
print(granite_final_metrics.to_string(index=False))

granite_final_metrics.to_csv(
    base / "granite_final_test_metrics.csv",
    index=False
)

print("\nAll Granite test metrics saved successfully!")


GRANITE ENGLISH ZERO-SHOT
              precision    recall  f1-score   support

       anger       0.52      0.34      0.42       125
anticipation       0.38      0.06      0.10       109
     disgust       1.00      0.02      0.03        63
        fear       0.44      0.24      0.31        70
         joy       0.46      0.13      0.21        82
     sadness       0.50      0.13      0.20        70
    surprise       0.32      0.11      0.17        81
       trust       0.32      0.10      0.15        83

   micro avg       0.45      0.15      0.23       683
   macro avg       0.49      0.14      0.20       683
weighted avg       0.48      0.15      0.21       683
 samples avg       0.14      0.15      0.14       683


GRANITE SPANISH ZERO-SHOT
              precision    recall  f1-score   support

       anger       0.59      0.21      0.31       125
anticipation       0.57      0.04      0.07       109
     disgust       0.50      0.03      0.06        63
        fear       0.45 

In [ ]:
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive")

# Load the two completed final test summaries
qwen = pd.read_csv(base / "qwen_final_test_metrics.csv")
granite = pd.read_csv(base / "granite_final_test_metrics.csv")

# Identify the model used
qwen.insert(0, "Model", "Qwen2.5-1.5B-Instruct")
granite.insert(0, "Model", "Granite-3.3-2B-Instruct")

# Combine the results
final_comparison = pd.concat(
    [qwen, granite],
    ignore_index=True
)

# Display a clean table
print("\nFINAL LLM TEST COMPARISON")
print(
    final_comparison[
        [
            "Model",
            "Experiment",
            "Micro-F1",
            "Macro-F1",
            "Jaccard",
            "Precision",
            "Recall",
            "Hamming Loss",
            "Exact Match",
            "Malformed Rate"
        ]
    ].to_string(index=False)
)

# Save to Google Drive
final_comparison.to_csv(
    base / "FINAL_LLM_TEST_COMPARISON.csv",
    index=False
)

print("\nFinal comparison saved successfully!")


FINAL LLM TEST COMPARISON
                  Model        Experiment  Micro-F1  Macro-F1  Jaccard  Precision   Recall  Hamming Loss  Exact Match  Malformed Rate
  Qwen2.5-1.5B-Instruct English zero-shot  0.336222  0.283232 0.280780   0.411890 0.284041      0.198240     0.190476        0.151139
  Qwen2.5-1.5B-Instruct Spanish zero-shot  0.310373  0.272085 0.269669   0.358238 0.273792      0.215062     0.175983        0.055901
  Qwen2.5-1.5B-Instruct English five-shot  0.342756  0.322799 0.302450   0.432071 0.284041      0.192547     0.219462        0.072464
  Qwen2.5-1.5B-Instruct Spanish five-shot  0.293286  0.275388 0.260007   0.369710 0.243045      0.207039     0.190476        0.070393
Granite-3.3-2B-Instruct English zero-shot  0.227074  0.197502 0.110421   0.446352 0.152269      0.183230     0.043478        0.057971
Granite-3.3-2B-Instruct Spanish zero-shot  0.186158  0.173006 0.088854   0.503226 0.114202      0.176501     0.049689        0.020704
Granite-3.3-2B-Instruct English fiv